# Scenario: The Weight Discrepancy Bug

In [19]:
import pandas as pd
import sqlite3
# crearting an Automated Scale Data (System A)
scale_data = {
    "patient_id": ["P-901", "P-902", "P-903", "P-904"],
    "scale_weight_kg": [70.0, 85.5, 62.1, 95.0]    
}
# adding dataset to DataFrame
df_scale_data = pd.DataFrame(scale_data)
# creating a Doctor EHR Manual Entries (System B)
ehr_data = {
    "patient_id":["P-901", "P-902", "P-903", "P-904"],
    "ehr_weight_kg":[70.0, 8.5, 62.1, 95.0] # look at "P-902"
}
# adding dataset to DataFrame
df_ehr_data = pd.DataFrame(ehr_data)
# creating sql to save dataframe on the temporary memory
connt = sqlite3.connect(":memory:")
df_scale_data.to_sql("scale_system", connt, index = False, if_exists = "replace")
df_ehr_data.to_sql("ehr_system", connt, index = False, if_exists = "replace")
# crteating function to run the query
def run_query(query):
    return pd.read_sql_query(query, connt)
print("********************************* Day 16 Cross-System Verification Database has been created! ***************** ")

********************************* Day 16 Cross-System Verification Database has been created! ***************** 


# The Conflict Detector

In [21]:
# query for all data to review
all_data1 = "SELECT * FROM scale_system"
print("******************************** all data1 for review ****************")
display(run_query(all_data1))
print()
all_data2 = "SELECT * FROM ehr_system"
print("******************************** all data2 for review ****************")
display(run_query(all_data2))
print()
# query that uses an INNER JOIN to link the scale_system and the ehr_system on patient_id. 
# Return the rows where scale_weight_kg does NOT equal (!= or <>) ehr_weight_kg
weight_mismatch = """
SELECT scale_system.patient_id, scale_system.scale_weight_kg, ehr_system.ehr_weight_kg FROM scale_system
INNER JOIN ehr_system
    ON scale_system.patient_id = ehr_system.patient_id
WHERE scale_system.scale_weight_kg != ehr_system.ehr_weight_kg
"""
print("********************************** the weight_mismatch between the 2 systems *********************")
display(run_query(weight_mismatch))

******************************** all data1 for review ****************


,patient_id,scale_weight_kg
0,P-901,70.0
1,P-902,85.5
2,P-903,62.1
3,P-904,95.0



******************************** all data2 for review ****************


,patient_id,ehr_weight_kg
0,P-901,70.0
1,P-902,8.5
2,P-903,62.1
3,P-904,95.0



********************************** the weight_mismatch between the 2 systems *********************


,patient_id,scale_weight_kg,ehr_weight_kg
0,P-902,85.5,8.5


# Calculating the Error Margin

In [23]:
# query that joins the tables and calculates the absolute difference between the two weights (ABS(scale_weight_kg - ehr_weight_kg)) as weight_error. 
# Filter it to only show errors greater than 5 kg.
error_margin = """
SELECT 
    scale_system.patient_id, 
    scale_system.scale_weight_kg, 
    ehr_system.ehr_weight_kg,
    ABS(scale_weight_kg - ehr_weight_kg) AS weight_error
FROM scale_system
INNER JOIN ehr_system ON scale_system.patient_id = ehr_system.patient_id
WHERE ABS(scale_weight_kg - ehr_weight_kg) > 5
"""
print("*********************************** error margin *************************")
display(run_query(error_margin))

*********************************** error margin *************************


,patient_id,scale_weight_kg,ehr_weight_kg,weight_error
0,P-902,85.5,8.5,77.0
